# MVCC Visibility, and the Bloat One Open Transaction Causes

Companion notebook for the [MVCC Snapshots and Tuple Visibility](https://codemia.ai/courses/database_internals/dbi_mvcc_snapshots_and_tuple_visibility) lesson on Codemia.

**What you will build**: a table that keeps every version it has ever had, a visibility rule that
decides which version each snapshot sees, and a cleanup process that can only remove what no
snapshot can still reach. By the end you will be able to explain why a table of ten thousand rows
can occupy a hundred thousand rows worth of space, and why the transaction responsible is usually
one that is doing nothing at all.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(13)

ROWS = 5000        # live rows in the table
STEPS = 60000      # updates applied over the run
VACUUM_EVERY = 500 # steps between cleanup passes

## Concept recap

Under multiversion concurrency control an update does not overwrite a row. It marks the current
version as ending at this transaction and writes a new version beginning at it, so a row is a chain
of versions each stamped with the transaction that created it and the transaction that superseded it.

A reader does not take locks. It takes a snapshot, which is a transaction number and the set of
transactions still running at that moment, and for each version it asks one question: was the
version created by a transaction this snapshot can see, and not yet superseded by one it can see.
Readers and writers therefore never wait for each other, which is the whole reason for the design.

The cost is that old versions have to be removed by something else, and that something else may only
remove a version once no snapshot can still reach it. That condition is a single number, the oldest
snapshot anyone is holding, and it is a minimum over every open transaction in the system. One
transaction that opened a snapshot an hour ago and has run no statements since holds that number
back for everybody, and nothing can be cleaned up past it, including in tables that transaction has
never touched.

## A table that keeps its versions

In [ ]:
class Table:
    """Each row is a list of (xmin, xmax) versions. xmax of None means this is the live one."""

    def __init__(self, rows=ROWS):
        self.chains = [[(0, None)] for _ in range(rows)]
        self.xid = 0
        self.removed = 0

    def update(self, row):
        self.xid += 1
        chain = self.chains[row]
        for i in range(len(chain) - 1, -1, -1):
            if chain[i][1] is None:
                chain[i] = (chain[i][0], self.xid)
                break
        chain.append((self.xid, None))
        return self.xid

    def visible(self, version, snapshot):
        """The one rule: created at or before the snapshot, not yet superseded at or before it."""
        xmin, xmax = version
        return xmin <= snapshot and (xmax is None or xmax > snapshot)

    def read(self, row, snapshot):
        chain = self.chains[row]
        for i, v in enumerate(chain):
            if self.visible(v, snapshot):
                return v, i + 1        # the version, and how many were examined to find it
        return None, len(chain)

    def versions(self):
        return sum(len(c) for c in self.chains)

    def vacuum(self, horizon):
        """Remove every superseded version that no snapshot at or after `horizon` can reach."""
        freed = 0
        for r, chain in enumerate(self.chains):
            keep = [v for v in chain if v[1] is None or v[1] > horizon]
            freed += len(chain) - len(keep)
            self.chains[r] = keep or [chain[-1]]
        self.removed += freed
        return freed

## The visibility rule on a table small enough to read

In [ ]:
demo = Table(rows=3)
demo.update(0)                 # xid 1
demo.update(1)                 # xid 2
snap_early = demo.xid          # a snapshot taken here sees the world as of xid 2
demo.update(0)                 # xid 3
demo.update(0)                 # xid 4
snap_late = demo.xid

print("row 0 version chain:", demo.chains[0])
print()
print("%-8s %-22s %-22s" % ("row", "snapshot at xid %d" % snap_early, "snapshot at xid %d" % snap_late))
for r in range(3):
    a, ca = demo.read(r, snap_early)
    b, cb = demo.read(r, snap_late)
    print("%-8d %-22s %-22s" % (r, "%s after %d checks" % (a, ca), "%s after %d checks" % (b, cb)))

Row 0 has four versions and three of them are dead. The early snapshot sees the one created by
transaction 1 and ended by transaction 3, because transaction 3 is in its future. The late snapshot
walks past three versions before it reaches the live one.

The number of checks is the hidden cost. Both snapshots are reading one row and getting one row
back, and one of them examined four versions to do it. A query plan will not show that, and a row
count will not show it either.

## What an open transaction costs the table

In [ ]:
def simulate(long_reader=None, steps=STEPS, skew=1.1, seed=13, vacuum_every=VACUUM_EVERY):
    """`long_reader` is (start_step, end_step) for one transaction that holds a snapshot."""
    gen = np.random.default_rng(seed)
    table = Table()
    targets = np.minimum(gen.zipf(skew, steps), ROWS) - 1
    held = None
    trace = []

    for step in range(steps):
        table.update(int(targets[step]))

        if long_reader is not None:
            start, end = long_reader
            if step == start:
                held = table.xid
            if step == end:
                held = None

        if step % vacuum_every == 0:
            horizon = table.xid if held is None else held
            table.vacuum(horizon)

        if step % 200 == 0:
            trace.append((step, table.versions()))

    return table, np.array(trace, dtype=float)


runs = {
    "no long transaction": simulate(),
    "one open for a third of the run": simulate(long_reader=(10000, 30000)),
    "one open for the whole run": simulate(long_reader=(10000, STEPS + 1)),
}

print("%-34s %10s %10s %12s" % ("scenario", "versions", "live rows", "bloat"))
for name, (table, _) in runs.items():
    print("%-34s %10d %10d %11.1fx" %
          (name, table.versions(), ROWS, table.versions() / ROWS))

In [ ]:
plt.figure(figsize=(9, 4.2))
colors = ["#5b8def", "#e8a33d", "#d1495b"]
for (name, (table, trace)), color in zip(runs.items(), colors):
    plt.plot(trace[:, 0] / 1000, trace[:, 1] / 1000, color=color, lw=2, label=name)
plt.axhline(ROWS / 1000, color="#8a8f98", ls="--", lw=1, label="live rows")
plt.xlabel("thousands of updates applied")
plt.ylabel("thousands of versions stored")
plt.title("the table, with and without something holding a snapshot")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The blue line is a healthy table. It rises between cleanup passes and falls back, and the sawtooth
stays close to the live row count forever.

The orange line is the same table with one transaction holding a snapshot for a third of the run.
Cleanup still runs on schedule and still removes nothing, because every version superseded since
that snapshot was taken is one the snapshot could still read. When the transaction finally ends, the
next pass reclaims all of it at once and the line drops back to where the blue one is.

The red line is the same thing with the transaction never ending. Nothing is wrong with the database,
cleanup is running, and the table grows without limit anyway.

Three details worth carrying out of this. The growth is not caused by the volume of updates; it is
caused by the oldest snapshot. The cleanup process is not failing; it is correctly refusing to remove
versions that are still reachable. And the recovery when the transaction ends is immediate, which is
why the first thing to look for on a bloating table is a long running or idle in transaction session
rather than a vacuum setting.

## What the versions cost a reader

In [ ]:
table, _ = runs["one open for the whole run"]
healthy, _ = runs["no long transaction"]

for name, t in [("healthy", healthy), ("bloated", table)]:
    lengths = np.array([len(c) for c in t.chains])
    checks = [t.read(r, t.xid)[1] for r in range(0, ROWS, 7)]
    print("%-10s mean chain %6.1f   longest %5d   versions examined per read %5.1f" %
          (name, lengths.mean(), lengths.max(), float(np.mean(checks))))

Reading the current version of a row means walking the chain until the visible version is found, so
the bloated table does several times the work for the same answer. The rows are the same rows and
the query is the same query.

This is why bloat shows up as a gradual, unattributable slowdown rather than as an error. Nothing
breaks, no plan changes, and every statement costs a little more than it did last week.

## Variations: cleanup frequency against the same workload

In [ ]:
print("%14s %12s %12s" % ("vacuum every", "versions", "bloat"))
for every in (100, 500, 2000, 10000):
    t, _ = simulate(vacuum_every=every)
    print("%14d %12d %11.1fx" % (every, t.versions(), t.versions() / ROWS))

Running cleanup more often keeps the table tighter, and the returns fall off quickly: the difference
between every hundred updates and every five hundred is small, and the difference between every five
hundred and every ten thousand is large.

Put that next to the previous section and the priority is clear. Tuning how often cleanup runs moves
a healthy table between one and a half times its live size and twice it. An open snapshot moves it
by whatever factor the workload can produce in the time nobody noticed. The settings are worth
getting right, and they cannot rescue a table from a transaction that will not end.

## What to try next

- Add deletes as versions with no successor and confirm that a deleted row is still readable by an
  older snapshot, which is why a delete frees nothing immediately either.
- Make the update distribution uniform rather than skewed and compare the longest chain. Skew
  concentrates the versions on a few rows, which is what makes a small hot table so much worse than
  its size suggests.
- Model the index as well: every new version needs an index entry, and the entry cannot be removed
  until the version can. Count index entries alongside versions and see which grows faster.
- Replace the single horizon with a per table horizon and see how much that would save, which is a
  real optimisation in recent engine versions and explains why an idle transaction in one database
  can stop hurting another.